# 🤖 Probando Google AI Studio (Gemini) Directo y con LangChain

Este notebook forma parte del curso **Introducción a la Inteligencia Artificial Generativa**.

Aquí aprenderás a interactuar con los modelos Gemini de Google de dos formas:
1. **SDK Oficial de Google GenAI (`google-genai`)**: La forma directa, moderna y oficial recomendada por Google.
2. **LangChain (`langchain-google-genai`)**: El framework estándar para construir aplicaciones avanzadas, cadenas (chains) y agentes.


## 📦 1. Instalación de dependencias

Si ejecutas este notebook por primera vez en un entorno nuevo, descomenta y ejecuta la siguiente celda para instalar los paquetes necesarios:


In [ ]:
# Descomenta la siguiente línea para instalar dependencias si es necesario:
# !pip install -q python-dotenv google-genai langchain-google-genai langchain-core


## 🔑 2. Carga y verificación de la API Key

Las claves de API deben mantenerse seguras y nunca compartirse en el código fuente ni subirse a Git.
Usamos `python-dotenv` para cargar las variables definidas en el archivo `.env` ubicado en la raíz del repositorio.

> **Nota:** Puedes obtener tu clave gratuita de Gemini en [Google AI Studio](https://aistudio.google.com/app/apikey).


In [2]:
import os
from dotenv import load_dotenv

# Cargar variables de entorno desde el archivo .env en la raíz del repositorio
# (../.env porque este notebook se encuentra en la subcarpeta 'notebooks/')
load_dotenv(dotenv_path="../.env")

# Obtenemos la clave de API
gemini_api_key = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")

if not gemini_api_key or "your_" in gemini_api_key:
    print("⚠️ ADVERTENCIA: No se encontró una clave válida en tu archivo .env.")
    print("1. Abre el archivo .env en la raíz del proyecto.")
    print("2. Pega tu API Key en GEMINI_API_KEY y GOOGLE_API_KEY.")
    print("3. Obtén tu clave en: https://aistudio.google.com/app/apikey")
else:
    # Sincronizamos ambas variables en el entorno para compatibilidad total
    os.environ["GEMINI_API_KEY"] = gemini_api_key
    os.environ["GOOGLE_API_KEY"] = gemini_api_key
    masked_key = gemini_api_key[:6] + "..." + gemini_api_key[-4:]
    print(f"✅ Clave de Gemini cargada exitosamente: {masked_key}")


✅ Clave de Gemini cargada exitosamente: AQ.Ab8...kB5w


---
## 🚀 Parte 1: Uso Directo con el SDK Oficial de Google GenAI (`google-genai`)

El SDK `google-genai` es el cliente oficial de Google para la API de Gemini. 
Automáticamente detecta la variable de entorno `GEMINI_API_KEY`.


### 1.1 Generación de texto simple
Haremos una consulta básica al modelo `gemini-3.6-flash` (rápido, multimodal y muy capaz).


In [7]:

from google import genai

# Inicializamos el cliente oficial
client = genai.Client()
chat = client.chats.create(model="gemini-3.5-flash")

prompt = "¿Cuáles son los 3 conceptos fundamentales que todo principiante debe comprender sobre la IA Generativa? Responde de forma concisa."

response = chat.send_message(prompt)

print("--- Respuesta de Gemini (SDK Directo) ---")
print(response.text)


Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


--- Respuesta de Gemini (SDK Directo) ---
Para un principiante, los 3 conceptos fundamentales para entender la IA Generativa son:

1. **Entrenamiento basado en Patrones (No es "magia", son datos):** 
La IA no piensa, no siente ni tiene conciencia. Aprende analizando enormes volúmenes de datos (textos, imágenes, código) para identificar **patrones y reglas** sobre cómo se estructuran las cosas. A partir de ahí, imita esos patrones para crear contenido nuevo.

2. **Generación Probabilística (Predicción, no búsqueda):** 
A diferencia de Google, que busca y copia información existente, la IA Generativa calcula matemáticamente **qué palabra, píxel o elemento es el más probable que deba ir después del anterior**. Debido a que es un juego de probabilidades y no una base de datos estática, la IA puede crear cosas originales, pero también puede cometer errores o inventar datos falsos (fenómeno conocido como **"alucinación"**).

3. **El *Prompt* (La instrucción clave):** 
Es el texto, imagen o c

### 1.2 Generación con Streaming (Tiempo real)
El *streaming* permite recibir los fragmentos de texto conforme el modelo los genera, mejorando drásticamente la experiencia del usuario final al reducir la latencia percibida.


In [ ]:
print("--- Generación con Streaming en tiempo real ---\n")

chat = client.chats.create(model="gemini-3.5-flash")

# Use send_message_stream for continuous chunk output
response_stream = chat.send_message_stream("Tell me a short poem about coding.")

print("AI: ", end="")
for chunk in response_stream:
    print(chunk.text, end="", flush=True)
print()

### 1.3 Configuración de Parámetros y System Instruction
Podemos personalizar el comportamiento del modelo utilizando:
- **`system_instruction`**: Define el rol, tono y restricciones del modelo.
- **`temperature`**: Controla la creatividad/aleatoriedad (0.0 = determinista y preciso, 1.0 = más creativo/variado).


In [ ]:
from google.genai import types

config = types.GenerateContentConfig(
    system_instruction="Eres un profesor universitario de Ciencias de la Computación amable, paciente y pedagógico.",
    temperature=0.3,
    max_output_tokens=300
)

chat = client.chats.create(model="gemini-3.5-flash", config=config)

response = chat.send_message(prompt)

print("--- Respuesta de Gemini (SDK Directo) ---")
print(response.text)


---
## 🦜️🔗 Parte 2: Uso a través de LangChain (`langchain-google-genai`)

LangChain es un ecosistema que permite conectar modelos con fuentes de datos externas, memorias, herramientas y agentes.
Aquí utilizaremos el componente `ChatGoogleGenerativeAI`.


### 2.1 Invocación básica con `ChatGoogleGenerativeAI`


In [10]:
from langchain_google_genai import ChatGoogleGenerativeAI

# Inicializamos el modelo de chat de LangChain
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    temperature=0.7
)

mensaje = "¿Qué es la técnica de Prompt Engineering y por qué es importante?"
respuesta = llm.invoke(mensaje)

print("--- Respuesta con LangChain ---")
print(respuesta.content)


Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


--- Respuesta con LangChain ---
[{'type': 'text', 'text': 'La **Ingeniería de Prompts** (o *Prompt Engineering* en inglés) es una de las disciplinas más relevantes y de mayor crecimiento en el ámbito de la Inteligencia Artificial (IA) generativa.\n\nA continuación, te explico detalladamente qué es, por qué es tan importante y cómo funciona.\n\n---\n\n### 1. ¿Qué es la Ingeniería de Prompts?\n\nEn términos sencillos, la **Ingeniería de Prompts es la práctica de diseñar, redactar y optimizar las instrucciones (llamadas "prompts") que se le dan a un modelo de Inteligencia Artificial** (como ChatGPT, Claude, Midjourney o DALL-E) para obtener la respuesta más precisa, útil y de alta calidad posible.\n\nUn "prompt" puede ser una pregunta, una frase, un fragmento de código o un texto detallado. \n\nNo se trata solo de "saber hablarle" a la IA, sino de entender cómo procesa la información el modelo para estructurar las instrucciones de manera que maximicen su rendimiento. Es, en esencia, **el 

### 2.2 Streaming con LangChain
LangChain implementa una interfaz uniforme `.stream()` en todos sus modelos.


In [ ]:
print("--- Streaming con LangChain ---\n")

for chunk in llm.stream("Dame 3 consejos clave para escribir prompts más efectivos:"):
    print(chunk.content, end="", flush=True)

print("\n\n[Streaming completado]")


### 2.3 Cadenas con LCEL (LangChain Expression Language) y Prompt Templates
Una de las mayores ventajas de LangChain es componer plantillas de prompts (`ChatPromptTemplate`), el modelo (`llm`) y parseadores de salida (`StrOutputParser`) en un flujo reutilizable usando el operador `|`.


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. Definimos una plantilla de prompt con variables dinámicas
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "Eres un tutor pedagógico especializado en Inteligencia Artificial. Explica temas de forma estructurada."),
    ("human", "Explica el concepto de '{tema}' adaptado a un público de nivel {nivel}. Limita tu respuesta a máximo 2 párrafos.")
])

# 2. Parseador que extrae únicamente la cadena de texto de la respuesta
parser = StrOutputParser()

# 3. Componemos la cadena LCEL (Prompt -> LLM -> Parser)
cadena = prompt_template | llm | parser

# 4. Invocamos la cadena pasando los parámetros
resultado = cadena.invoke({
    "tema": "Embeddings (Incrustaciones Vectoriales)",
    "nivel": "principiante sin conocimientos técnicos previos"
})

print("--- Resultado de la cadena LCEL ---")
print(resultado)


## 🎯 Conclusión y Próximos Pasos
¡Felicitaciones! Has probado exitosamente:
1. La llamada directa a Gemini usando el SDK oficial `google-genai`.
2. El uso de streaming y configuración de `system_instruction` y `temperature`.
3. La integración con LangChain mediante `ChatGoogleGenerativeAI`.
4. La construcción de cadenas reutilizables con LCEL (`ChatPromptTemplate | llm | StrOutputParser`).

En el siguiente notebook probaremos **Groq Cloud** para experimentar la inferencia ultra rápida con modelos abiertos como LLaMA 3.
